# __Inteligencia Artificial - IUA 2026__

### *Exploración y Análisis del Dataset*

### Integrantes
* **Celeste Antonella Badami** — cbadami845@alumnos.iua.edu.ar
* **Benjamin Vargas** — bvargas161@alumnos.iua.edu.ar

---


## 1. Introducción
Este documento analiza el dataset de métricas con el fin de evaluar la calidad de los datos y definir las decisiones de diseño para el análisis exploratorio (EDA) y la ingeniería de features del modelo de ETA.

## 2. Preparación del entorno

In [1]:
# importación de librerias
import pandas as pd

In [2]:
# Carga de la muestra de datos preeliminar
df = pd.read_csv('../data/sample.csv')

## 3. Análisis General

In [3]:
# Cabecera de la muestra de datos
df.head()

,id,bearing,current_status,direction_id,occupancy_status,latitude,longitude,bus_id,route_id,route_pattern_id,...,precipitation,relative_humidity_2m,weather_code,is_day,current_speed,free_flow_speed,current_travel_time,freeflow_travel_time,confidence,road_closure
0,1,315.0,IN_TRANSIT_TO,0,FULL,42.359920,-71.094600,y1923,1,1-_-0,...,0.0,65.0,2.0,True,18,18,216,216,1.0,False
1,2,113.0,IN_TRANSIT_TO,1,MANY_SEATS_AVAILABLE,42.371286,-71.116519,y1918,1,1-_-1,...,0.0,63.0,2.0,True,18,18,125,125,1.0,False
2,3,43.0,IN_TRANSIT_TO,1,MANY_SEATS_AVAILABLE,42.349551,-71.088831,y1878,1,1-_-1,...,0.0,63.0,2.0,True,27,33,129,105,1.0,False
3,4,135.0,IN_TRANSIT_TO,1,MANY_SEATS_AVAILABLE,42.342210,-71.084290,y1831,1,1-_-1,...,0.0,65.0,2.0,True,12,15,105,84,1.0,False
4,5,145.0,STOPPED_AT,1,MANY_SEATS_AVAILABLE,42.339168,-71.080314,y1830,1,1-_-1,...,0.0,65.0,2.0,True,17,19,252,225,1.0,False


In [10]:
# Información general
df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 2132 entries, 0 to 2131
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     2132 non-null   int64  
 1   bearing                1653 non-null   float64
 2   current_status         2132 non-null   str    
 3   direction_id           2132 non-null   int64  
 4   occupancy_status       2131 non-null   str    
 5   latitude               2132 non-null   float64
 6   longitude              2132 non-null   float64
 7   bus_id                 2132 non-null   str    
 8   route_id               2132 non-null   int64  
 9   route_pattern_id       2132 non-null   str    
 10  shape_id               2132 non-null   int64  
 11  agency_id              2132 non-null   int64  
 12  line_id                2132 non-null   str    
 13  distancia_en_traza     2132 non-null   float64
 14  distancia_fuera_traza  2132 non-null   float64
 15  es_desvio      

### 3.1 Valores Nulos

Se verifica que las variables fundamentales para la geolocalización y temporalidad (latitude, longitude, date_time) no presentan valores faltantes. De igual forma, las métricas de tránsito se encuentran completas. Por el contrario, existen registros nulos en algunas variables meteorológicas secundarias y, principalmente, en la columna bearing (dirección en grados del vehículo), la cual concentra la mayor cantidad de datos faltantes.

In [11]:
# Analisis de faltantes/ nulos

# Crear un DataFrame con el análisis de nulos
vacios_summary = pd.DataFrame({
    'Tipo_Dato': df.dtypes,
    'Valores_Vacios': df.isnull().sum(),
    'Valores_Validos': df.notnull().sum(),
    'Porcentaje_Vacio (%)': (df.isnull().sum() / len(df) * 100).round(2)
})

# Filtrar solo las columnas que tienen al menos un valor vacío y ordenar de mayor a menor
vacios_detallados = vacios_summary[vacios_summary['Valores_Vacios'] > 0].sort_values(
    by='Valores_Vacios', ascending=False
)

vacios_detallados

,Tipo_Dato,Valores_Vacios,Valores_Validos,Porcentaje_Vacio (%)
bearing,float64,479,1653,22.47
temperature_2m,float64,3,2129,0.14
precipitation,float64,3,2129,0.14
weather_code,float64,3,2129,0.14
relative_humidity_2m,float64,3,2129,0.14
is_day,object,3,2129,0.14
occupancy_status,str,1,2131,0.05


Esta tabla nos muestra que muy pocas son las perdidas de datos en terminos de porcentaje (<1%)

Bearing tiene tambien un porcentaje porcentual bajo de perdida

### 3.2 La Flag "*es_desvio*"

![Vehiculo desviado](../images/desvio.jpeg)

El bot cosechador de datos incluía una funcionalidad para calcular la distancia entre el vehículo y la traza del tramo. Se estableció un umbral máximo de 70 metros para determinar cuándo el vehículo se ha desviado del trayecto. Dado que las desviaciones puntuales (como paradas intermedias) no son constantes y representan outliers o valores atípicos en este dominio, es necesario procesar y filtrar dichos registros para evitar distorsionar las métricas y la estimación en tiempo real.

In [14]:
total_filas = len(df)
filas_desviadas = (df['es_desvio'] == 1).sum()
porcentaje_desvio = (filas_desviadas / total_filas) * 100

print(f"Total de registros: {total_filas}")
print(f"Registros con desvío: {filas_desviadas}")
print(f"Porcentaje de desvíos: {porcentaje_desvio:.2f}%")

Total de registros: 2132
Registros con desvío: 85
Porcentaje de desvíos: 3.99%


El bajo porcentaje de desvíos registrado confirma una alta adherencia de los vehículos a la traza oficial, lo que ratifica la utilidad y calidad de la gran mayoría de los datos para el entrenamiento del modelo.

### 3.3 Estatus posibles (*"current_status"*)

Se busca determinar los valores posibles para la feature de *current_status*

In [ ]:
#